# Network Science (981G5) Assessment

Abstract\Introduction:
> Some text here

#### Contents
- [Heading 1]()
- [Heading 2]()

---

#### Notebook Setup

In [ ]:
# ========================
# Installations
# ========================

# Install required dependencies
%pip install -q --upgrade pip
%pip install -q pandas statsbombpy networkx matplotlib jinja2 seaborn scipy

# Enable live auto-reloading for helpers.py updates
%load_ext autoreload
%autoreload 2

In [ ]:
# ========================
# Imports
# ========================

from typing import Optional, Tuple, Dict, List, Callable
import warnings
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from statsbombpy import sb
from statsbombpy.api_client import NoAuthWarning


# Custom helper module
import helper as hp

# Global notebook configurations
warnings.simplefilter("ignore", NoAuthWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

### Other

## 1. Statbomb API: Raw Data Import
This the section that works on importing the raw events data from the Statsbomb and processing it in the form which is suitable for network analysis. Events data refers to identifable on-field actions, i.e. pass, tackle, shot. 

---

Section Contents: 
- [Heading 1]()
- [Heading 2]()

---

#### 1.1 Competition Extraction
StatBomb structure their data is various tables and granularly layers. We want to work with the 2023/2024 FA Women's Super League (`competition_id: 37`, `season_id: 281`) data. The code below queries to the `competitions` table to confirm that we have the correct codes. 

In [ ]:
# Retrieve the competitions
competitions_df = sb.competitions()
print(f"Total Competitions Returned: {len(competitions_df)}")

# Filter the DataFrame for WSL 2023/2024
target_filter = (competitions_df['competition_id'] == 37) & (competitions_df['season_id'] == 281)
wsl_competition = competitions_df[target_filter].to_dict(orient='records')[0]

# Print the extracted row
print("\nExtracted Competition Payload:")
print(wsl_competition)

In [ ]:
COMPETITION_ID = 37
SEASON_ID = 281

### 1.2 Match, Team, Player and Events Extractions
To construct a Passing Network (PassMap), we need to extract and aggregate event-level data across individual matches. Each match yields two networks for the competing teams. In order to compile these team-match network we need to compile the nesecary information to structure our network and call the API for the specific information. 

1. **Nodes (Players):** Extracted from match lineups to determine player identity, tactical position, and appearance duration.
2. **Edges (Passes):** Extracted from match events to capture successfully completed passes between teammates.
3. **Graph Attributes:** Extracted to store tactical context, such as starting formations.

The following code offload processing to utilities held in `helper.py`.

| Function | Primary Purpose | Output/Pipeline Role |
| :--- | :--- | :--- |
| `hp.fetch_match_details()` | Downloads event stream and lineup payloads in a single API call. | Captures match duration (`max_minute`) and raw payloads. |
| `hp.extract_team_roster()` &<br>`hp.extract_11_players()` | Parses substitution timestamps and calculates individual appearances. | Computes total minutes played and isolates the core 11 players with highest volume. |
| `hp.extract_team_formation()` | Queries tactical setups (e.g., `4-3-3`, `3-4-3`) from starting XI events. | Attaches tactical system metadata to the team network. |
| `hp.extract_successful_passes()` | Filters out incomplete or intercepted passes. | Returns completed pass actions needed to build adjacency matrices. |

The result is a list of match-team entries containing: match id, team name, starting formation, team total passes, the full player roster including subs and the main team of 11 players who played the most minutes. This gives us all of the information be needed to query the API, pull the pass data and construct a network.

In [ ]:
# Extract full seasons worth of match_ids
matches_df = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
matches = matches_df["match_id"].to_list()
print(f"WSL 23/24 contains {len(matches)} matches.")

match_records = []

for match in matches:
    events, lineups, max_minute = hp.fetch_match_details(match)
    for team_name, lineup_df in lineups.items():

        # Roster extraction 
        roster = hp.extract_team_roster(lineup_df, max_minute)
        top_11_players = hp.extract_11_players(roster)

        # Tactical and event extraction
        formation = hp.extract_team_formation(events, team_name)
        team_passes = hp.extract_successful_passes(events, team_name)
    
        match_records.append({
            "match_id": match,
            "team": team_name,
            "formation": formation,
            "total_passes": len(team_passes),
            "full_roster": roster,
            "top_11_players": top_11_players
        })

In [ ]:
print(f"{len(match_records)} team-match records extracted")
print(40*"=")
print("Sample[0]: ")
print("Match ID:", match_records[0]["match_id"])
print("Team Name:", match_records[0]["team"])
print("Formation:", match_records[0]["formation"])
print("Team Passes:", match_records[0]["total_passes"])
print("Full Roster", len(match_records[0]["full_roster"]))
print("Most Used Players:", len(match_records[0]["top_11_players"]))
print("Most Used Players:", match_records[0]["top_11_players"])
print("Top 3 Players in Roster:")
for p in match_records[0]["top_11_players"][:3]:
    print(
        f"  • {p['Player Name']} ({p['Position']}) - {p['Minutes Played']} mins"
    )


### 1.3 League Statistics
The code below uses the compile match records to pull some league level statistics. Of note we see that there are 105,262 accross the whole seasons, 264 networks (match-team) with an average of 399 passes which ranges from 120 to 847. This is important because passes accumulate to become the edges in our network.

In [ ]:
stats_dict = hp.calculate_league_summary_stats(match_records)

# ==============================================================================
# PRINT GENERATED TABLE STATS
# ==============================================================================
print("\n" + "=" * 60)
print("EXTRACTED SUMMARY STATISTICS (WSL 2023/2024)")
print("=" * 60)
print(f"Total Matches Analyzed:                 {stats_dict['total_matches']}")
print(f"Total Team Match Networks:              {stats_dict['total_team_games']}")
print(f"Total Completed Season Passes:          {stats_dict['total_passes']:,}")
print(f"Mean Passes per Team per Match:         {stats_dict['mean_passes']:.1f} "
f"(Range: {stats_dict['min_passes']} - {stats_dict['max_passes']})")
print(f"Mean Unique Players Used per Game:      {stats_dict['mean_players_used']:.1f} "
f"(Range: {stats_dict['min_players_used']} - {stats_dict['max_players_used']})")
print("=" * 60)

### 1.4 Highest Match-Team Pass Total
The network we will focus on for this project is the match with the highest total of passes recorded by a single team. This match is of note because it is likely to have very few games, if any, which have a similar number of passes. We know that the number of passes impact the network topology and properties of PassMap, therefore, to formally analyse this network we need other networks with similar passes, otherwise the number of passes will dominate the topology. Therefore, this network is a new candidate and demonstrates why we will need null models for analysis.

In [ ]:
# Returns a tuple: (list_index, max_record_dict)
max_idx, highest_pass_match = max(
    enumerate(match_records), 
    key=lambda item: item[1]["total_passes"]
)

print(f"Record Index: {max_idx}")
print(f"Match ID:     {highest_pass_match['match_id']}")
print(f"Team:         {highest_pass_match['team']}")
print(f"Passes:       {highest_pass_match['total_passes']}")

## 2. Network
The Network section outlines the underlying infrastructure and connectivity framework, as well as, visualisation tooling.

---

Section Contents: 
- [Heading 1]()
- [Heading 2]()

---

#### 2.1 Build Network Function
To transform raw relational event data into formal graph representations, we construct a dedicated pipeline function, `build_passmap_network()`. This pipeline processes individual raw event instances from scratch to dynamically construct an attribute-enriched, directed, and weighted graph ($G \in \mathbb{R}^{V \times E}$)
- `extract_top11_pass_events()`: Filters the raw event log down to completed passes belonging strictly to the target team and the 11 modeled players.
- `compute_player_average_positions()`: Calculates each player's mean pitch coordinates $(\bar{x}, \bar{y})$ from their pass events, binding them to nodes as spatial attributes.
- `aggregate_pass_edges()`: Aggregates pairwise pass counts to assign directed edge weights (e.g., 30 completed passes from Player A to Player B yields an edge weight of 30).

The output is a weighted, direct network with node attributes. 

In [ ]:
def build_passmap_network(match_record: dict, events_df: Optional[pd.DataFrame] = None) -> nx.DiGraph:
    """Constructs a weighted, directed NetworkX graph (nx.DiGraph) for a team passmap.
    
    Parameters
    ----------
    match_record : dict
        A team-match dictionary containing 'match_id', 'team', and 'top_11_players'.
    events_df : pd.DataFrame, optional
        Pre-loaded events DataFrame for the match. If None, fetches directly via API.
            
    Returns
    -------
    nx.DiGraph
        Directed passmap graph.
    """
    # Compile the Match-Team records
    m_id = match_record["match_id"]
    team_name = match_record["team"]
    top_11_players = match_record["top_11_players"]
    top_11_ids = {p['Player ID'] for p in top_11_players}
    player_id_to_name = {p['Player ID']: p['Player Name'] for p in top_11_players}
    
    # API Fallback
    if events_df is None:
        events_df = sb.events(match_id=m_id)

    # Extract raw pass (Edges) and Position (Node) information
    passes_df = hp.extract_top11_pass_events(events_df, team_name, top_11_ids)
    avg_locations = hp.compute_player_average_positions(passes_df)
    
    G = nx.DiGraph(
    team_name=team_name,
    total_passes=match_record["total_passes"]
    )

    # Construct the network nodes from the 11 players
    for p in top_11_players:
        p_id, p_name, p_pos = p['Player ID'], p['Player Name'], p['Position']
        loc = avg_locations.get(p_id, {'x': 50.0, 'y': 50.0})
        
        G.add_node(
            p_name,
            player_id=p_id,
            position=p_pos,
            pos=(loc['x'], loc['y']),
            x=loc['x'],
            y=loc['y']
        )
        
    edges = hp.aggregate_pass_edges(passes_df, player_id_to_name)
    for passer, recipient, weight in edges:
        G.add_edge(passer, recipient, weight=weight)
        
    return G

#### 2.2 Highest Pass Match Network
`G_highest` holds the PassMap network for the highest pass match

In [ ]:
# Pass the dictionary record directly
target_record = match_records[max_idx]
G_highest = build_passmap_network(target_record)

#### 2.3 Basic Network Visualisation
Here is an extremely basic visualisation of the network. While the format for the visualisation doesn't matter for most network properties, given we are analysing a spatially constrained game it is useful to see the network well formatted. In this implementation, the nodes are plotted as per their average pass position during the game. This basic implementation doesn't plot the edge weight

In [ ]:
# plt.figure(figsize=(4, 3))

# # Extract Node Locations
# pos = nx.get_node_attributes(G_highest, 'pos')

# # Draw basic graph
# nx.draw(G_highest, pos=pos)

# plt.title("Basic PassMap Network Topology")
# plt.axis("on")
# plt.grid(True)
# plt.show()

#### 2.4 Pitch Plotting Underlay
We introduce a custom pitch-underlay function (`draw_vertical_pitch()`) to visualize network nodes over a 2D spatial grid. While purely cosmetic with no effect on underlying graph metrics, rendering nodes at their mean pitch coordinates significantly improves readability and visual appeal. This helper was built natively with Matplotlib to avoid bloated third-party dependencies such as `mlpsoccer`.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))

# Draw Pitch Underlay
hp.draw_vertical_pitch(ax=ax)

#### 2.5 Plotting Raw Passes
Here I am plotting the raw passes for a match-team and then an individual player within the match. This is demonstate just how messy and unintuative pass data is and why network analysis is so valuable.

In [ ]:

# Create a figure with 1 row and 2 columns
# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RAW_TEAM = 'Arsenal WFC'
# RAW_NAME = 'Emily Ann Fox'
# RAW_MATCH = sb.events(match_id=3913160)

# # Plot the individual player passes on the first subplot
# hp.plot_player_raw_passes(RAW_MATCH, RAW_TEAM, RAW_NAME, ax=axes[0])
# axes[0].set_title('Emily Ann Fox')

# # Plot all team passes on the second subplot
# hp.plot_player_raw_passes(RAW_MATCH, RAW_TEAM, None, ax=axes[1])
# axes[1].set_title('Arsenal WFC (All Passes)')

# plt.tight_layout()
# plt.show()

#### 2.6 Pitch Plot PassMap
This section plots the network overlayed on the pitch. Typically, this is the most intuitive way to plot PassMaps as it gives us insight as to specific areas of the pitch that players operate in and provides us insight as to why certain relationships, clusters and pass routes may be forming. However, this network is very dense with pass volume and as a result makes it difficult to observe the edges. 

In [ ]:
# 1. Unpack fig and ax explicitly
# fig, ax = plt.subplots(figsize=(6.5, 9.0), facecolor='#ffffff')

# # 2. Call the passmap function with the explicitly assigned ax
# hp.plot_passmap_on_pitch(G_highest, ax=ax)

# plt.show()

#### 2.7 Frameless Pitch Plot
This visual retains the spatial coordinates but maximise the space in the images framing. 

In [ ]:
# Declare the figure window
# fig, ax = plt.subplots(figsize=(6.5, 9.0), facecolor='#ffffff')

# # Plot frameless graph
# hp.plot_passmap_frameless(G_highest, ax=ax)

# # Remove all external margins
# plt.tight_layout(pad=0.5)
# plt.show()

#### 2.8 Filtered PassMap
As the PassMap represents cumulative passes across an entire match it can be good to set a threshold from which edges are plotted by. This is partly done to improve the view of the network by "removing" edges. But functionally, this removes edges which don't represent any sort of relationship. Two players on the opposite sides of the pitch may make a single interaction in a defensive situation whereby their true positional roles have been compromised. Doing this intensifies stronger links. Note, this is generally just a cosmetic action, we don't tend to actually want to remove edges from the network itself.

In [ ]:
# Filter graph to only show pass combinations made at least 8 times
# G_threshold_8 = hp.filter_graph_edges(G_highest, min_weight=5)

# # Declare the figure window
# fig, ax = plt.subplots(figsize=(6.5, 9.0), facecolor='#ffffff')
# # hp.plot_passmap_on_pitch(G=G_threshold_8, ax=ax)
# hp.plot_passmap_frameless(G_threshold_8, ax=ax)
# plt.tight_layout(pad=0.5)
# plt.show()

### 2.9 Adjacency Matrix

In [ ]:
import networkx as nx
import pandas as pd

# Convert to a labeled Pandas DataFrame
adj_df = nx.to_pandas_adjacency(G_highest, weight='weight', dtype=int)

print(adj_df)

adj_matrix = nx.to_numpy_array(G_highest, weight='weight')

print(adj_matrix)

## 3. Degree Analysis
To evaluate the structural balance and individual workloads within the passing network, we implement the `analyze_degree_and_heterogeneity()` pipeline. This function extracts both micro-level (player) and macro-level (team) **degree** network properties from the directed pass graph ($G$).

In [94]:
# Analyze graph using the integrated function
player_df, macro_stats, top_hubs_df = hp.analyze_degree_and_heterogeneity(G_highest, top_n_hubs=5)

### 3.1 Micro-Level Player Metrics:
- Calculates unweighted in/out-degree ($k_{in}, k_{out}$) to measure the diversity of passing partners
- weighted in/out-strength ($s_{in}, s_{out}$) to quantify total passes received and completed.
- Net Flow ($\Delta s_i = s_{out} - s_{in}$) measures directional asymmetry to identify net distributors ($\Delta s_i > 0$) versus net receivers ($\Delta s_i < 0$).
- Pass Ratio ($s_{out} / s_{in}$) to idetnify positional roles, distinguish build-up playmakers ($> 1.0$) from target finishing endpoints 

In [95]:
# 1. Player-Level Table
print("\n" + "="*90)
print("####Micro-Level Execution: Player-Level Degree Metrics (Arsenal WFC)")
print("="*90)
print(player_df.to_string())


####Micro-Level Execution: Player-Level Degree Metrics (Arsenal WFC)
                                          Position  k_in (In-Degree)  k_out (Out-Degree)  s_in (Passes Rec.)  s_out (Passes Comp.)  Total Volume (s_tot)  Net Flow (Δs_i)  Pass Ratio (s_out / s_in)
Player                                                                                                                                                                                              
Carlotte Wubben-Moy               Left Center Back                 9                  10                 120                   128                   248                8                       1.07
Kim Little                 Left Defensive Midfield                10                   9                 117                   123                   240                6                       1.05
Leah Williamson                  Right Center Back                 8                  10                  99                   103            

### 3.2 Macro-Level Network Heterogeneity
At the team level, we evaluate workload dispersion and network centralization:

| Metric | Notation | Formula / Code Variable | What It Measures |
| :--- | :--- | :--- | :--- |
| **Mean Unweighted Degree** | $\langle k \rangle$ | `mean_k` | Measures the average number of unique passing connections per player across the team. |
| **Degree Variance** | $\text{Var}(k)$ | `var_k` | Captures the spread of unique passing connections around the team mean. |
| **Degree Standard Deviation** | $\sigma_k$ | `std_k` | Quantifies the average dispersion of player connection counts from the squad average. |
| **Second Moment** | $\langle k^2 \rangle$ | `second_moment` | Measures the second moment of the unweighted degree distribution to emphasize the presence of highly connected hubs. |
| **Coefficient of Variation** | $CV_k$ | `cv_k` ($\sigma_k / \langle k \rangle$) | Measures unweighted degree heterogeneity to assess how unevenly unique passing channels are distributed across all players. |


In [92]:
# 2. Macro Network Metrics
print("\n" + "="*60)
print("Macro-Level Metrics: Centralization & Network Heterogeneity")
print("="*60)
for metric_name, val in macro_stats.items():
    print(f"{metric_name:<45}: {val:.4f}")


Macro-Level Metrics: Centralization & Network Heterogeneity
Team Node Volume Variance Var(s_tot)         : 5681.9008
Mean Unweighted Degree <k>                   : 16.1818
Degree Variance Var(k)                       : 7.7851
Degree Std Dev σ_k                           : 2.7902
Second Moment <k^2>                          : 269.6364
Coefficient of Variation (CV_k)              : 0.1724
Normalized Second Moment (<k^2> / <k>)       : 16.6629


### Hub Identification 
Ranks the top $N$ players by total pass volume ($s_{tot}$) and calculates their **Relative Volume** ($s_i / \langle s_{tot} \rangle$) to highlight the team's primary playmakers relative to the squad average.

In [93]:
# 3. Top Network Hubs
print("\n" + "="*60)
print("Top Network Hubs (Rank-Ordered)")
print("="*60)
print(top_hubs_df.to_string(index=False))


Top Network Hubs (Rank-Ordered)
 Rank                 Player                 Position  Total Volume (s_tot)  k_in (In-Degree)  k_out (Out-Degree)  Relative Volume (s_i / <s_tot>)
    1    Carlotte Wubben-Moy         Left Center Back                   248                 9                  10                             1.89
    2             Kim Little  Left Defensive Midfield                   240                10                   9                             1.83
    3        Leah Williamson        Right Center Back                   202                 8                  10                             1.54
    4        Victoria Pelova Right Defensive Midfield                   179                 9                   8                             1.37
    5 Stephanie-Elise Catley                Left Back                   160                 9                   9                             1.22


## 4. Average Shortest Path